# Game Intelligence AI — Model Benchmark v2 (Auto GPU + BitsAndBytes Fallback)

This version addresses the previous benchmark failure. The uploaded report showed `bitsandbytes==0.50.2`, but Transformers could not use 4-bit quantization. This notebook tests the actual `bitsandbytes` import and CUDA backend before loading models.

**Recovery strategy**
1. Detect the GPU and available VRAM automatically.
2. Diagnose the installed BitsAndBytes package and backend.
3. Use 4-bit quantization only if the backend test succeeds.
4. If 4-bit is unavailable, fall back to FP16 for a small model only when VRAM allows it; skip larger models rather than risking an out-of-memory crash.
5. Save diagnostics and benchmark results to `/content/GameIntelligenceAI`.

No Google Drive is mounted or used. `/content` is temporary. GitHub remains for source/notebooks; Hugging Face Hub is for larger model/data artifacts subject to quotas and licenses.

## 0. Runtime setup

In Colab select **Runtime → Change runtime type → GPU** if available. Do not change PyTorch or CUDA manually.

Run cells in order. If a diagnostic cell fails, it records the actual exception instead of hiding it behind a generic message.

In [ ]:
import os, sys, json, time, gc, shutil, subprocess, platform, importlib, importlib.util
from pathlib import Path
from datetime import datetime, timezone
PROJECT_ROOT=Path("/content/GameIntelligenceAI")
for d in ["benchmarks","logs","tmp","models/cache"]:
    (PROJECT_ROOT/d).mkdir(parents=True,exist_ok=True)
print("Python:",sys.version)
print("Platform:",platform.platform())
print("Project:",PROJECT_ROOT)
print("Google Drive dependency: NONE")

## 1. Detect GPU and disk

In [ ]:
import torch
GPU_AVAILABLE=torch.cuda.is_available()
GPU_NAME=torch.cuda.get_device_name(0) if GPU_AVAILABLE else None
GPU_VRAM_GB=(torch.cuda.get_device_properties(0).total_memory/1024**3) if GPU_AVAILABLE else 0.0
CUDA_VERSION=torch.version.cuda
disk_total,disk_used,disk_free=shutil.disk_usage("/content")
print("GPU available:",GPU_AVAILABLE)
print("GPU:",GPU_NAME)
print("VRAM GB:",round(GPU_VRAM_GB,2))
print("PyTorch:",torch.__version__,"| CUDA build:",CUDA_VERSION)
print("Free disk GB:",round(disk_free/1024**3,2))
p=subprocess.run(["bash","-lc","nvidia-smi"],capture_output=True,text=True)
print(p.stdout if p.returncode==0 else p.stderr)

## 2. Diagnose BitsAndBytes — do not reinstall yet

The previous report listed BitsAndBytes 0.50.2. The error may mean that the package is installed but its Python module or CUDA backend cannot be imported. This cell checks:
- package metadata version;
- Python module location;
- direct import;
- CUDA backend diagnostics where supported;
- Transformers' own availability check.

The full exception is saved in a JSON file.

In [ ]:
import importlib.metadata as metadata
bnb_diag={"distribution_version":None,"module_spec":None,"direct_import":"NOT TESTED",
          "bnb_version":None,"transformers_available":None,"cuda_available":GPU_AVAILABLE,
          "cuda_build":CUDA_VERSION,"errors":[]}
try:
    bnb_diag["distribution_version"]=metadata.version("bitsandbytes")
except Exception as e:
    bnb_diag["errors"].append("metadata: "+repr(e))
try:
    spec=importlib.util.find_spec("bitsandbytes")
    bnb_diag["module_spec"]=None if spec is None else spec.origin
except Exception as e:
    bnb_diag["errors"].append("find_spec: "+repr(e))
try:
    import bitsandbytes as bnb
    bnb_diag["direct_import"]="OK"
    bnb_diag["bnb_version"]=getattr(bnb,"__version__","unknown")
    print("Direct bitsandbytes import: OK")
    print("Imported version:",bnb_diag["bnb_version"])
except Exception as e:
    bnb_diag["direct_import"]="FAILED"
    bnb_diag["errors"].append("direct_import: "+repr(e))
    print("Direct bitsandbytes import FAILED:",type(e).__name__,str(e))
try:
    from transformers.utils import is_bitsandbytes_available
    bnb_diag["transformers_available"]=bool(is_bitsandbytes_available())
    print("Transformers sees bitsandbytes available:",bnb_diag["transformers_available"])
except Exception as e:
    bnb_diag["errors"].append("transformers_check: "+repr(e))
    print("Transformers availability check failed:",repr(e))

## 3. Test an actual 4-bit CUDA operation

A successful Python import alone does not prove that the CUDA kernels can run. This cell attempts a small 4-bit quantized linear layer. If this fails, the detailed exception is stored and the benchmark switches to a safe fallback.

In [ ]:
bnb_cuda_ok=False
bnb_cuda_error=None
if GPU_AVAILABLE and bnb_diag["direct_import"]=="OK":
    try:
        import bitsandbytes as bnb
        layer=bnb.nn.Linear4bit(64,32,bias=False,compute_dtype=torch.float16,
                                compress_statistics=True,quant_type="nf4").cuda()
        x=torch.randn(2,64,device="cuda",dtype=torch.float16)
        with torch.inference_mode():
            y=layer(x)
        assert y.shape==(2,32)
        bnb_cuda_ok=True
        del layer,x,y
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print("4-bit CUDA smoke test: PASSED")
    except Exception as e:
        bnb_cuda_error=f"{type(e).__name__}: {str(e)}"
        bnb_diag["errors"].append("cuda_smoke: "+bnb_cuda_error)
        print("4-bit CUDA smoke test FAILED:",bnb_cuda_error)
else:
    print("4-bit CUDA smoke test skipped: GPU or direct import unavailable.")
bnb_diag["cuda_smoke_test"]=bnb_cuda_ok
bnb_diag["cuda_smoke_error"]=bnb_cuda_error
DIAG_PATH=PROJECT_ROOT/"logs"/"bitsandbytes_diagnostics.json"
DIAG_PATH.write_text(json.dumps(bnb_diag,indent=2,ensure_ascii=False))
print("Diagnostic report:",DIAG_PATH)

## 4. Select the quantization strategy automatically

- If direct import and the 4-bit CUDA smoke test both pass, use 4-bit for eligible candidates.
- If they fail, use FP16 only for the 3B candidate when GPU VRAM is at least 12 GB.
- On smaller GPUs or without a GPU, skip model inference rather than loading an unsafe model size.

This avoids repeatedly failing every candidate with the same quantization error.

In [ ]:
USE_4BIT=bool(GPU_AVAILABLE and bnb_diag["direct_import"]=="OK" and bnb_cuda_ok)
if USE_4BIT:
    STRATEGY="4BIT"
elif GPU_AVAILABLE and GPU_VRAM_GB>=12:
    STRATEGY="FP16_SMALL_MODEL_ONLY"
else:
    STRATEGY="NO_SAFE_MODEL_LOAD"

if not GPU_AVAILABLE:
    MAX_NEW_TOKENS=96
elif GPU_VRAM_GB<12:
    MAX_NEW_TOKENS=128
elif GPU_VRAM_GB<20:
    MAX_NEW_TOKENS=160
elif GPU_VRAM_GB<32:
    MAX_NEW_TOKENS=256
else:
    MAX_NEW_TOKENS=384

MAX_INPUT_TOKENS=2048 if GPU_VRAM_GB<20 else 4096
print("Selected strategy:",STRATEGY)
print("Max new tokens:",MAX_NEW_TOKENS)
print("Max input tokens:",MAX_INPUT_TOKENS)

## 5. Candidate models

On a 16GB-class GPU, 3B is the safe fallback if quantization is broken. Larger candidates are attempted only when the 4-bit CUDA test succeeds. Candidate availability and licensing must be checked against each official model card before commercial use.

In [ ]:
CANDIDATES=[
 {"id":"Qwen/Qwen2.5-Coder-3B-Instruct","label":"Qwen2.5-Coder 3B","size":"small"},
 {"id":"Qwen/Qwen2.5-Coder-7B-Instruct","label":"Qwen2.5-Coder 7B","size":"medium"},
 {"id":"Qwen/Qwen3-4B","label":"Qwen3 4B","size":"small"},
 {"id":"Qwen/Qwen3-8B","label":"Qwen3 8B","size":"medium"},
]
if STRATEGY=="4BIT":
    SELECTED=CANDIDATES if GPU_VRAM_GB>=12 else [c for c in CANDIDATES if c["size"]=="small"]
elif STRATEGY=="FP16_SMALL_MODEL_ONLY":
    SELECTED=[CANDIDATES[0]]
else:
    SELECTED=[]
print("Strategy:",STRATEGY)
print("Selected:",[c["id"] for c in SELECTED])
print("Skipped:",[c["id"] for c in CANDIDATES if c not in SELECTED])

## 6. Optional Hugging Face token

Public repositories usually do not need a token. For gated/private models, add a Colab Secret named `HF_TOKEN`. Do not paste or print tokens.

In [ ]:
HF_TOKEN=None
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:
    pass
print("HF token configured:",bool(HF_TOKEN))

## 7. Small game-development task set

In [ ]:
TASKS=[
 {"id":"web_platformer","prompt":"Write a self-contained JavaScript updatePlayer(player,input,dt) for a 2D platformer. Include delta-time movement, gravity, jump only when grounded. Return code only.","checks":["function","gravity","ground"]},
 {"id":"godot_pickup","prompt":"Write Godot 4 GDScript for a coin Area2D that detects a player, prevents double collection, emits a signal and calls queue_free. Return code only.","checks":["Area2D","signal","queue_free"]},
 {"id":"unity_health","prompt":"Write a minimal Unity C# MonoBehaviour health component with max health, damage, clamp at zero and a death event. Return code only.","checks":["MonoBehaviour","Mathf","event"]},
 {"id":"debug_js","prompt":"Explain and fix: for (var i=0;i<3;i++){setTimeout(()=>console.log(i),0)}. Provide corrected code and reason.","checks":["let","0","1","2"]},
 {"id":"level_design","prompt":"Design an original beginner 2D puzzle-platformer level. Include objective, mechanics, layout, hazards, accessibility and measurable completion condition.","checks":["objective","mechanic","hazard","completion"]}
]
TASK_LIMIT=2

## 8. Load model with selected strategy

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM
def cleanup(model=None,tokenizer=None):
    try: del model
    except: pass
    try: del tokenizer
    except: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def load_candidate(model_id,size):
    tok=AutoTokenizer.from_pretrained(model_id,token=HF_TOKEN)
    kw={"token":HF_TOKEN}
    if USE_4BIT:
        from transformers import BitsAndBytesConfig
        kw.update({
          "device_map":"auto",
          "torch_dtype":torch.float16,
          "quantization_config":BitsAndBytesConfig(
            load_in_4bit=True,bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True)
        })
    else:
        if size!="small" or GPU_VRAM_GB<12:
            raise RuntimeError("FP16 fallback is restricted to small models on GPUs with >=12GB VRAM.")
        kw.update({"device_map":"auto","torch_dtype":torch.float16,
                   "low_cpu_mem_usage":True})
    model=AutoModelForCausalLM.from_pretrained(model_id,**kw)
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token=tok.eos_token
    return tok,model

## 9. Run a bounded benchmark

The default runs two prompts per model. The report records load failures and actual exceptions. If a model fails to load, the notebook continues to the next selected candidate.

In [ ]:
RESULTS=[]
def format_prompt(tok,text):
    messages=[{"role":"user","content":text}]
    if getattr(tok,"chat_template",None):
        return tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    return text

for cand in SELECTED:
    rec={"model_id":cand["id"],"label":cand["label"],"loaded":False,
         "tasks":[],"error":None,"strategy":STRATEGY}
    tok=model=None
    try:
        if GPU_AVAILABLE: torch.cuda.reset_peak_memory_stats()
        t0=time.time()
        tok,model=load_candidate(cand["id"],cand["size"])
        rec["load_seconds"]=round(time.time()-t0,2)
        rec["loaded"]=True
        for task in TASKS[:TASK_LIMIT]:
            prompt=format_prompt(tok,task["prompt"])
            inp=tok(prompt,return_tensors="pt",truncation=True,max_length=MAX_INPUT_TOKENS)
            inp={k:v.to(model.device) for k,v in inp.items()}
            start=time.time()
            with torch.inference_mode():
                out=model.generate(**inp,max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,pad_token_id=tok.pad_token_id)
            elapsed=time.time()-start
            new=out[0,inp["input_ids"].shape[1]:]
            answer=tok.decode(new,skip_special_tokens=True)
            score=sum(check.lower() in answer.lower() for check in task["checks"])
            rec["tasks"].append({
              "task":task["id"],"seconds":round(elapsed,2),
              "tokens":int(new.numel()),
              "tokens_per_second":round(new.numel()/max(elapsed,0.001),2),
              "simple_checks":f"{score}/{len(task['checks'])}",
              "output":answer[:3500]
            })
            print(cand["label"],task["id"],"speed",rec["tasks"][-1]["tokens_per_second"],
                  "tok/s | checks",rec["tasks"][-1]["simple_checks"])
        if GPU_AVAILABLE:
            rec["peak_vram_gb"]=round(torch.cuda.max_memory_allocated()/1024**3,2)
    except Exception as e:
        rec["error"]=f"{type(e).__name__}: {str(e)[:1400]}"
        print("FAILED:",cand["id"],rec["error"])
    finally:
        cleanup(model,tok)
    RESULTS.append(rec)
    (PROJECT_ROOT/"benchmarks"/"partial_results.json").write_text(
        json.dumps(RESULTS,indent=2,ensure_ascii=False))

## 10. Summarize results

In [ ]:
SUMMARY=[]
for r in RESULTS:
    scores=[]; speeds=[]
    for t in r.get("tasks",[]):
        try:
            a,b=t["simple_checks"].split("/")
            scores.append(int(a)/int(b))
        except Exception: pass
        speeds.append(t.get("tokens_per_second",0))
    SUMMARY.append({
      "model":r["model_id"],"loaded":r["loaded"],
      "tasks_completed":len(r["tasks"]),
      "mean_simple_check":round(sum(scores)/len(scores),3) if scores else None,
      "mean_tokens_per_second":round(sum(speeds)/len(speeds),2) if speeds else None,
      "peak_vram_gb":r.get("peak_vram_gb"),"error":r.get("error")
    })
print(json.dumps(SUMMARY,indent=2,ensure_ascii=False))

## 11. Save diagnostics and benchmark report

Download the report at the end. `/content` is temporary, so keep a copy outside the runtime.

In [ ]:
REPORT={
 "project":"Game Intelligence AI",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "hardware":{"gpu_available":GPU_AVAILABLE,"gpu_name":GPU_NAME,
             "gpu_vram_gb":round(GPU_VRAM_GB,2),"cuda":CUDA_VERSION,
             "torch":torch.__version__,"free_disk_gb_start":round(disk_free/1024**3,2)},
 "bitsandbytes_diagnostics":bnb_diag,
 "strategy":STRATEGY,
 "results":RESULTS,"summary":SUMMARY,
 "limitations":["Small prompt set, not comprehensive.",
 "String checks are heuristic, not semantic evaluation.",
 "Inference success does not prove QLoRA training will fit.",
 "Verify model card, license and dataset provenance before commercial use."]
}
REPORT_PATH=PROJECT_ROOT/"benchmarks"/"model_benchmark_report_v2.json"
REPORT_PATH.write_text(json.dumps(REPORT,indent=2,ensure_ascii=False))
print("Saved:",REPORT_PATH)
try:
    from google.colab import files
    files.download(str(REPORT_PATH))
except Exception as e:
    print("Colab download helper unavailable:",e)

## 12. Next step

Send `model_benchmark_report_v2.json` here. If the BitsAndBytes CUDA test failed, the diagnostic section will tell us the underlying error. Do not begin fine-tuning until the model load and training memory requirements have been checked.

No Google Drive is used. Keep notebooks/source in GitHub. Store large datasets, adapters and checkpoints in a persistent store such as Hugging Face Hub only within its current storage, bandwidth and license conditions. `/content` is temporary.

In [ ]:
print("Completed.")
print("GPU profile:",GPU_NAME,round(GPU_VRAM_GB,2),"GB")
print("Quantization strategy:",STRATEGY)
print("Models attempted:",len(RESULTS))
print("Successful loads:",sum(r["loaded"] for r in RESULTS))
print("Report:",REPORT_PATH)